# E1 · Clasificador de contexto (P1) → §9.1

**Pregunta (P1):** ¿el clasificador asigna el contexto correcto y rechaza lo que no reconoce, también fuera de EVA?

Protocolo: §8.5.1 de la memoria y §6.1 de `experimentos.md`. No usa LLM ni API: solo el ResNet-18
desplegado (`models/clasificador_contexto/`) y la clase real del sistema, `ContextClassifier`
(`tools/cnn_context.py`).

| Parte | Conjunto | Qué mide | Necesita |
|---|---|---|---|
| **E1a** | C1 (`test.csv`, 725) | Exactitud sin umbral, P/R/F1 por clase, matriz. Comprobación de integridad contra la fase 5 | — |
| **E1b** | C1 + O (`otro.csv`, 273) | Curva riesgo-cobertura, AUROC C1 frente a O y valores en el umbral 0,85 | — |
| **E1c** | val (725) + O | La misma curva en `val.csv`: ¿0,85 es un artefacto de haberlo elegido mirando C1? | — |
| **E1d** | C2 (`data/`, 100) | Reparto de etiquetas, % a `otro` y cambio de dominio de la confianza. Con A1: exactitud con 6 etiquetas y matriz 6×6 | A1 para la parte con verdad |
| **E1e** | T100 | ¿El orquestador (LLM) copia bien la etiqueta de `predict()`? | Tanda T100 |
| **Grad-CAM** | C1 | Un acierto y un error del par más confundido (producto → arquitectura) | — |

### Cómo usarlo
1. Kernel: el Python de **`.venv`**. Ejecuta en orden.
2. La primera vez infiere unas 1.800 imágenes (GPU: 1-3 minutos) y guarda las predicciones en
   `experimentos/resultados/E1/predicciones_*.csv`. Después las reutiliza (pon `RECALCULAR = True`
   si cambian los pesos o los splits).
3. Escribe solo en `experimentos/resultados/E1/`, en `outputs/experimentos/E1/` (Grad-CAM de
   `predict()`) y, si no existe, crea la plantilla `experimentos/anotaciones/A1_contexto.csv`.
4. Las partes que dependen de algo pendiente (A1, T100, criterios de C2) no fallan: se informan
   como **pendientes**.

## 1. Configuración y criterios

Los criterios se leen de `experimentos/config_experimentos.yaml`, donde se fijan **antes** de ejecutar.
Un criterio a `null` sale como «pendiente».

In [7]:
import sys, json, time, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ / "experimentos") not in sys.path:
    sys.path.insert(0, str(RAIZ / "experimentos"))
import comun
comun.preparar_sys_path()

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from IPython.display import display, Markdown

from tfg_multiagente_fotografia.tools import cnn_context as cc     # <- EL CÓDIGO REAL

RECALCULAR = False

CFG = comun.cargar_config()
CFG_E1 = CFG["E1"]
REF = CFG_E1["referencia"]
CRIT = CFG_E1["criterios"]
torch.manual_seed(CFG["semilla"]); np.random.seed(CFG["semilla"])

EVA = comun.ruta_config("eva_proyecto")
EVA_IMG = comun.ruta_config("eva_imagenes")
SPLITS = EVA / "data" / "splits"
MODELOS = RAIZ / "models" / "clasificador_contexto"
ANOT = comun.ruta_config("anotaciones")
RES = comun.dir_resultados("E1")

# Paleta de las figuras: categórica en orden fijo (C1, O, C2/val) y rampa secuencial azul.
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA2, GRIS, REJILLA, EJE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
CMAP = LinearSegmentedColormap.from_list(
    "azul", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11,
    "axes.titlecolor": TINTA, "axes.labelcolor": TINTA2, "axes.edgecolor": EJE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": REJILLA, "grid.linewidth": 0.6, "xtick.color": GRIS, "ytick.color": GRIS,
    "lines.linewidth": 2, "legend.frameon": False,
})

def guardar_figura(fig, nombre):
    fig.savefig(RES / nombre, bbox_inches="tight")
    plt.show()

def pct(x):
    return f"{100 * x:.1f} %"

pd.DataFrame(
    [("F1 mínima por clase en C1 ≥", CRIT["f1_min_por_clase_C1"], "descriptivo: ya se conocía (0,7687)"),
     ("Exactitud sobre aceptadas en C2 ≥", CRIT["exactitud_aceptadas_min_C2"], "fijar antes de anotar A1"),
     ("% de C2 enviado a «otro» ≤", CRIT["pct_otro_max_C2"], "fijar antes de ejecutar")],
    columns=["criterio", "valor fijado", "nota"])

,criterio,valor fijado,nota
0,F1 mínima por clase en C1 ≥,0.75,"descriptivo: ya se conocía (0,7687)"
1,Exactitud sobre aceptadas en C2 ≥,0.85,fijar antes de anotar A1
2,% de C2 enviado a «otro» ≤,20.00,fijar antes de ejecutar


## 2. Procedencia y congelación

Se guarda el commit, las versiones y el MD5 de los tres artefactos del clasificador. Si todavía no
existe `experimentos/congelacion.json` (lo genera `00_preparacion`), el notebook avisa: los números
valen como prueba, pero no son los definitivos del capítulo 9.

El `assert` comprueba que los pesos del sistema son el checkpoint `fase4_fine_tuning_ligero.pt`
evaluado en la fase 5.

In [8]:
PROC = comun.procedencia(sorted(p for p in MODELOS.iterdir() if p.is_file()))
CONGELADO = comun.comprobar_congelacion()

md5_pesos = PROC["md5"]["models/clasificador_contexto/pesos_fine_tuning_ligero.pt"]
assert md5_pesos == REF["md5_pesos"], f"Los pesos no son los de la fase 5: {md5_pesos}"

print(f"commit {PROC['commit'][:7]} · árbol limpio: {PROC['arbol_limpio']} · congelado: {CONGELADO}")
print(f"GPU: {PROC['gpu']} · CUDA {PROC['cuda']} · torch {PROC['versiones']['torch']}")
print(f"MD5 pesos: {md5_pesos} (= fase 5)")

TypeError: comprobar_congelacion() missing 1 required positional argument: 'actual'

## 3. Datos y anotaciones

- **C1, O, val:** los splits de la fase 1 (estratificados 70/15/15, `SEED = 42`). Las imágenes se leen
  de `EVA_together/` sin copiarlas al repositorio. Se comprueba el tamaño, que no se solapan y que
  existe cada fichero.
- **C2:** `data/imagen{1..100}.jpg`.
- **A1** (contexto verdadero de C2, 6 etiquetas): `experimentos/anotaciones/A1_contexto.csv`. Si no
  existe se crea la plantilla vacía. **No se rellena con la predicción**, para no sesgar al anotador.

In [9]:
class_to_idx = json.loads((MODELOS / "class_to_idx.json").read_text(encoding="utf-8"))
CLASES = [c for c, _ in sorted(class_to_idx.items(), key=lambda kv: kv[1])]
assert CLASES == comun.ETIQUETAS_SISTEMA

SPLITS_CSV = {"C1": "test", "O": "otro", "val": "val"}
splits = {k: pd.read_csv(SPLITS / f"{v}.csv") for k, v in SPLITS_CSV.items()}
for k, df in splits.items():
    assert len(df) == REF[f"n_{k}"], f"{k}: {len(df)} filas"
    faltan = [i for i in df.image_id if not (EVA_IMG / f"{i}.jpg").exists()]
    assert not faltan, f"{k}: faltan {len(faltan)} imágenes en {EVA_IMG}"
assert set(splits["C1"].label) == set(CLASES) == set(splits["val"].label)
assert set(splits["O"].label) == {"otro"}
ids = {k: set(df.image_id) for k, df in splits.items()}
assert not (ids["C1"] & ids["O"] or ids["C1"] & ids["val"] or ids["val"] & ids["O"]), "splits solapados"

C2 = comun.imagenes_C2()
assert len(C2) == 100, f"C2 tiene {len(C2)} imágenes"

reparto = pd.concat({k: df.label.value_counts() for k, df in splits.items()}, axis=1).fillna(0).astype(int)
reparto.loc["total"] = reparto.sum()
reparto

,C1,O,val
label,,,
retrato-humano,155,0,155
paisaje,152,0,152
arquitectura,143,0,142
producto-still_life,139,0,140
animal,136,0,136
otro,0,273,0
total,725,273,725


In [10]:
RUTA_A1 = ANOT / "A1_contexto.csv"
if not RUTA_A1.exists():
    ANOT.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"imagen": [p.stem for p in C2], "contexto": "", "notas": ""}).to_csv(
        RUTA_A1, index=False, encoding="utf-8")
    print(f"Creada la plantilla {RUTA_A1.relative_to(RAIZ)} (valores: {', '.join(comun.ETIQUETAS_A1)})")

A1 = comun.leer_csv_robusto(RUTA_A1).fillna("")
A1["contexto"] = A1["contexto"].astype(str).str.strip()
invalidas = A1[(A1.contexto != "") & ~A1.contexto.isin(comun.ETIQUETAS_A1)]
assert invalidas.empty, f"Etiquetas A1 no válidas:\n{invalidas}"
N_A1 = int((A1.contexto != "").sum())
A1_COMPLETA = N_A1 == len(C2) and set(A1.imagen) == {p.stem for p in C2}
print(f"A1: {N_A1}/{len(C2)} imágenes anotadas · completa: {A1_COMPLETA}")

A1: 100/100 imágenes anotadas · completa: True


## 4. Inferencia

Se instancia `ContextClassifier` con los mismos tres artefactos que usa `ContextClassifierTool`. La
inferencia por lotes usa **el mismo** `clf.model`, `clf.transform` (Resize 256 → CenterCrop 224 →
normalización ImageNet) y la misma función de apertura del sistema (`_abrir_para_clasificar`). Se
guardan por imagen los logits, las probabilidades, la clase de mayor probabilidad y la etiqueta que
daría el sistema (`otro` si la confianza < umbral).

Después se comprueba la réplica contra `clf.predict()`, el método que llama el sistema, sobre las 100
de C2 y 25 de C1. Así la inferencia por lotes queda garantizada como equivalente a la del sistema.

**Tolerancia.** En GPU, PyTorch usa por defecto TF32 en las convoluciones de cuDNN
(`torch.backends.cudnn.allow_tf32 = True`), y el algoritmo depende del tamaño de lote. Por eso la
confianza de un lote de 64 y la de `predict()` (lote 1) difieren hasta unos 5e-4. Con TF32 desactivado
coinciden hasta 4e-7. Se mantiene TF32 porque es como corre el sistema (y como se midió la fase 5), y se
acepta una diferencia < `TOL_TF32 = 1e-3`. Las imágenes con la confianza a menos de esa distancia del
umbral pueden cambiar de etiqueta entre ejecuciones: se listan aparte (dato para E4 y §8.6).

In [11]:
clf = cc.ContextClassifier(
    weights_path=MODELOS / "pesos_fine_tuning_ligero.pt",
    class_to_idx_path=MODELOS / "class_to_idx.json",
    threshold_path=MODELOS / "umbral.json",
    gradcam_dir=RAIZ / "outputs" / "experimentos" / "E1" / "gradcam_predict",
)
UMBRAL = clf.threshold
TOL_TF32 = 1e-3
assert [clf.idx_to_class[i] for i in range(len(CLASES))] == CLASES


class ImagenesPorRuta(Dataset):
    """Abre y transforma cada imagen exactamente como `ContextClassifier.predict()`."""
    def __init__(self, rutas):
        self.rutas = [Path(r) for r in rutas]
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        return clf.transform(cc._abrir_para_clasificar(self.rutas[i]))


def inferir(rutas, lote=64):
    logits = []
    with torch.no_grad():
        for xb in DataLoader(ImagenesPorRuta(rutas), batch_size=lote, shuffle=False, num_workers=0):
            logits.append(clf.model(xb.to(clf.device)).float().cpu())
    logits = torch.cat(logits)
    return logits.numpy(), torch.softmax(logits, dim=1).numpy()


def predicciones(nombre, ids, rutas, etiquetas=None):
    """Tabla por imagen (id, etiqueta, predicción, confianza, probabilidades, logits), en caché."""
    ruta = RES / f"predicciones_{nombre}.csv"
    if ruta.exists() and not RECALCULAR:
        return pd.read_csv(ruta)
    t0 = time.time()
    logits, probs = inferir(rutas)
    df = pd.DataFrame({"id": list(ids)})
    if etiquetas is not None:
        df["etiqueta"] = list(etiquetas)
    df["prediccion"] = [CLASES[i] for i in probs.argmax(1)]
    df["confianza"] = probs.max(1)
    df["etiqueta_sistema"] = np.where(df.confianza < UMBRAL, "otro", df.prediccion)
    for j, c in enumerate(CLASES):
        df[f"p_{c}"] = probs[:, j]
    for j, c in enumerate(CLASES):
        df[f"logit_{c}"] = logits[:, j]
    df.to_csv(ruta, index=False, encoding="utf-8")
    print(f"{nombre}: {len(df)} imágenes en {time.time() - t0:.0f} s -> {ruta.relative_to(RAIZ)}")
    return df


def rutas_eva(df):
    return [EVA_IMG / f"{i}.jpg" for i in df.image_id]


PRED = {k: predicciones(k, df.image_id, rutas_eva(df), df.label if k != "O" else None)
        for k, df in splits.items()}
PRED["C2"] = predicciones("C2", [p.stem for p in C2], C2)
{k: len(v) for k, v in PRED.items()}

{'C1': 725, 'O': 273, 'val': 725, 'C2': 100}

In [12]:
# Réplica frente al sistema: clf.predict() imagen a imagen (también genera su Grad-CAM).
muestra_C1 = splits["C1"].sample(25, random_state=CFG["semilla"])
casos = [(p.stem, p, "C2") for p in C2] + [(i, EVA_IMG / f"{i}.jpg", "C1") for i in muestra_C1.image_id]
filas = []
for id_, ruta, conj in casos:
    r = clf.predict(str(ruta))
    filas.append({"conjunto": conj, "id": id_, "label_predict": r["label"], "conf_predict": r["confidence"]})
chk = pd.DataFrame(filas)
lotes = pd.concat([PRED["C2"].assign(conjunto="C2"), PRED["C1"].assign(conjunto="C1")])
lotes["id"] = lotes["id"].astype(str)
chk["id"] = chk["id"].astype(str)
chk = chk.merge(lotes[["conjunto", "id", "etiqueta_sistema", "confianza"]], on=["conjunto", "id"])
assert len(chk) == len(casos)

dif_conf = (chk.conf_predict - chk.confianza).abs().max()
junto_al_umbral = (chk.confianza - UMBRAL).abs() < TOL_TF32
assert dif_conf < TOL_TF32, f"La confianza por lotes difiere de predict(): {dif_conf:.2e}"
assert (chk.label_predict == chk.etiqueta_sistema)[~junto_al_umbral].all()
print(f"Réplica OK en {len(chk)} imágenes · diferencia máxima de confianza: {dif_conf:.1e} "
      f"· etiquetas distintas: {(chk.label_predict != chk.etiqueta_sistema).sum()}")

# Imágenes cuya etiqueta podría cambiar por la precisión TF32 (confianza a menos de TOL del umbral).
cerca_umbral = {k: PRED[k].loc[(PRED[k].confianza - UMBRAL).abs() < TOL_TF32, ["id", "confianza"]]
                for k in PRED}
{k: v.to_dict("records") for k, v in cerca_umbral.items()}

Réplica OK en 125 imágenes · diferencia máxima de confianza: 4.4e-04 · etiquetas distintas: 0


{'C1': [{'id': 333949, 'confianza': 0.84957874},
  {'id': 168621, 'confianza': 0.8496195}],
 'O': [{'id': 10188, 'confianza': 0.8508944},
  {'id': 864074, 'confianza': 0.8505303}],
 'val': [{'id': 957556, 'confianza': 0.8508493}],
 'C2': []}

## 5. E1a · C1 sin umbral

Exactitud con la clase de mayor probabilidad (sin `otro`). Es la cifra **limpia** de C1: el checkpoint
se eligió con `val.csv` y C1 no intervino. El `assert` exige reproducir la exactitud y la matriz de la
fase 5: si falla, ha cambiado algo (pesos, transform o splits).

In [ ]:
c1 = PRED["C1"]
exactitud_C1 = accuracy_score(c1.etiqueta, c1.prediccion)
k_aciertos = int((c1.etiqueta == c1.prediccion).sum())
ic_exactitud_C1 = comun.intervalo_wilson(k_aciertos, len(c1))
reporte_C1 = classification_report(c1.etiqueta, c1.prediccion, labels=CLASES, output_dict=True, zero_division=0)
cm_C1 = confusion_matrix(c1.etiqueta, c1.prediccion, labels=CLASES)

assert round(exactitud_C1, 4) == REF["exactitud_C1"], exactitud_C1
assert (cm_C1 == np.array(REF["matriz_C1"])).all(), cm_C1
print(f"Integridad OK: exactitud {exactitud_C1:.4f} y matriz idénticas a la fase 5")
print(f"Exactitud C1 = {exactitud_C1:.4f} (IC 95 % Wilson {ic_exactitud_C1[0]:.3f}-{ic_exactitud_C1[1]:.3f})")

tabla_clases = pd.DataFrame(reporte_C1).T.loc[CLASES + ["macro avg"], ["precision", "recall", "f1-score", "support"]]
tabla_clases["support"] = tabla_clases["support"].astype(int)
tabla_clases.round(4)

In [ ]:
def abreviar(c):
    return {"producto-still_life": "producto", "retrato-humano": "retrato"}.get(c, c)


def dibujar_matriz(cm, filas, columnas, titulo, nombre):
    """Matriz de confusión: color = proporción por fila (recall), texto = recuento."""
    prop = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(1.0 * len(columnas) + 2, 0.8 * len(filas) + 1.4))
    ax.imshow(prop, cmap=CMAP, vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if prop[i, j] > 0.55 else TINTA)
    ax.set_xticks(range(len(columnas)), [abreviar(c) for c in columnas], rotation=30, ha="right")
    ax.set_yticks(range(len(filas)), [abreviar(c) for c in filas])
    ax.set_xlabel("Predicción"); ax.set_ylabel("Verdadera"); ax.set_title(titulo, loc="left")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    guardar_figura(fig, nombre)


dibujar_matriz(cm_C1, CLASES, CLASES, f"C1 · sin umbral · exactitud {exactitud_C1:.3f}", "matriz_C1.png")

confusiones = (pd.DataFrame(cm_C1, index=CLASES, columns=CLASES).stack()
               .rename_axis(["verdadera", "predicha"]).rename("n").reset_index())
confusiones = confusiones[confusiones.verdadera != confusiones.predicha].nlargest(3, "n")
confusiones

## 6. E1b · C1 + O con umbral

Con el umbral *t*, una imagen se **acepta** si su confianza (probabilidad máxima) es ≥ *t*; si no, va a
`otro`. Sobre C1 se mide:

- **cobertura:** fracción aceptada. El 27,4 % de la fase 5 es `1 − cobertura`: **no** es «mal
  rechazado», porque incluye aciertos y fallos;
- **exactitud sobre las aceptadas** (exactitud selectiva) y su complemento, el **riesgo selectivo**;
- **errores capturados:** fracción de los fallos de C1 que el umbral manda a `otro` (rechazar un fallo es
  bueno);
- **aciertos perdidos:** fracción de los aciertos que el umbral manda a `otro` (el coste real).

Sobre O se mide el **% rechazado** (el 64,8 %). El **AUROC** C1 frente a O, con la confianza como
puntuación, resume la separación sin depender de ningún umbral.

⚠️ El umbral 0,85 se eligió mirando estas mismas curvas de C1 y O (fase 5, celda 18). Por eso las cifras
de C1 en 0,85 son **optimistas**. E1c lo controla con val.

In [ ]:
c = CFG_E1["curva_umbrales"]
UMBRALES = np.round(np.arange(c["inicio"], c["fin"] + c["paso"] / 2, c["paso"]), 2)


def curva(validas, otro, umbrales):
    conf = validas.confianza.to_numpy()
    acierto = (validas.prediccion == validas.etiqueta).to_numpy()
    conf_o = otro.confianza.to_numpy()
    filas = []
    for t in umbrales:
        acepta = conf >= t
        exactitud_sel = acierto[acepta].mean() if acepta.any() else np.nan
        filas.append({
            "umbral": float(t),
            "cobertura": acepta.mean(),
            "rechazadas_pct": 100 * (~acepta).mean(),
            "exactitud_aceptadas": exactitud_sel,
            "riesgo_selectivo": 1 - exactitud_sel,
            "errores_capturados": (~acepta & ~acierto).sum() / max((~acierto).sum(), 1),
            "aciertos_perdidos": (~acepta & acierto).sum() / max(acierto.sum(), 1),
            "O_rechazado_pct": 100 * (conf_o < t).mean(),
        })
    return pd.DataFrame(filas)


def en_umbral(tabla, t):
    fila = tabla[np.isclose(tabla.umbral, t)]
    assert len(fila) == 1, f"El umbral {t} no está en la rejilla"
    return fila.iloc[0]


def auroc(validas, otro):
    y = np.r_[np.ones(len(validas)), np.zeros(len(otro))]
    return roc_auc_score(y, np.r_[validas.confianza, otro.confianza])


curva_C1 = curva(PRED["C1"], PRED["O"], UMBRALES)
en_085_C1 = en_umbral(curva_C1, UMBRAL)
auroc_C1_O = auroc(PRED["C1"], PRED["O"])

assert round(en_085_C1.rechazadas_pct, 1) == REF["C1_rechazado_pct"], en_085_C1.rechazadas_pct
assert round(en_085_C1.O_rechazado_pct, 1) == REF["O_rechazado_pct"], en_085_C1.O_rechazado_pct
print(f"Integridad OK: en {UMBRAL} se reproducen el {REF['C1_rechazado_pct']} % y el {REF['O_rechazado_pct']} % de la fase 5")
print(f"AUROC C1 frente a O = {auroc_C1_O:.4f}")

n_err_C1 = int((c1.etiqueta != c1.prediccion).sum())
display(Markdown(
    f"**En el umbral {UMBRAL}:** cobertura {pct(en_085_C1.cobertura)} · exactitud sobre las aceptadas "
    f"{en_085_C1.exactitud_aceptadas:.4f} (frente a {exactitud_C1:.4f} sin umbral) · errores capturados "
    f"{pct(en_085_C1.errores_capturados)} de {n_err_C1} · aciertos perdidos {pct(en_085_C1.aciertos_perdidos)} · "
    f"O rechazado {en_085_C1.O_rechazado_pct:.1f} %"))
curva_C1.set_index("umbral").loc[CFG_E1["umbrales_ganancia_marginal"]].round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Izquierda: qué manda a «otro» cada umbral, separado por población.
series = [("aciertos_perdidos", "C1 · aciertos rechazados", AZUL, 100),
          ("errores_capturados", "C1 · errores rechazados", NARANJA, 100),
          ("O_rechazado_pct", "O · rechazadas", AQUA, 1)]
for col, etiqueta, color, escala in series:
    ax1.plot(curva_C1.umbral, curva_C1[col] * escala, color=color, label=etiqueta)
ax1.axvline(UMBRAL, color=GRIS, lw=1, ls="--")
ax1.text(UMBRAL, 102, f" {UMBRAL}", color=TINTA2, fontsize=8)
ax1.set(xlabel="Umbral de confianza", ylabel="% enviado a «otro»", ylim=(0, 105), xlim=(0.5, 1.0))
ax1.set_title("Qué rechaza cada umbral", loc="left")
ax1.legend(loc="upper left", fontsize=8)

# Derecha: riesgo-cobertura (exactitud sobre las aceptadas frente a cobertura).
ax2.plot(curva_C1.cobertura * 100, curva_C1.exactitud_aceptadas, color=AZUL, label="C1")
ax2.plot(en_085_C1.cobertura * 100, en_085_C1.exactitud_aceptadas, "o", ms=8, color=AZUL, mec="white", mew=2)
ax2.annotate(f"t = {UMBRAL}", (en_085_C1.cobertura * 100, en_085_C1.exactitud_aceptadas),
             xytext=(8, -12), textcoords="offset points", fontsize=8, color=TINTA2)
ax2.axhline(exactitud_C1, color=GRIS, lw=1, ls="--")
ax2.text(2, exactitud_C1, f"sin umbral ({exactitud_C1:.3f})", va="bottom", fontsize=8, color=TINTA2)
ax2.set(xlabel="Cobertura (% de C1 aceptado)", ylabel="Exactitud sobre las aceptadas", xlim=(0, 102))
ax2.set_title("Riesgo-cobertura en C1", loc="left")
fig.tight_layout()
guardar_figura(fig, "curva_umbral_C1_O.png")

## 7. E1c · Control del umbral con val + O

Se repite la curva sobre `val.csv`, que no intervino en la elección del umbral (sí en la del
checkpoint). También se repite el **análisis de ganancia marginal** de la fase 5 (celda 19), con el que
se razonó 0,85:

- beneficio = Δ % de O rechazado;
- coste = Δ % de válidas rechazadas;
- la fase 5 tomó como óptimo estricto el final del tramo con mayor ratio (0,80) y lo subió a 0,85
  mientras el ratio siguiera > 1.

Si con val la regla lleva al mismo sitio, 0,85 no es un artefacto de C1.

O es el mismo en C1 y en val: el control solo cambia la parte de las imágenes válidas.

In [ ]:
curva_val = curva(PRED["val"], PRED["O"], UMBRALES)
en_085_val = en_umbral(curva_val, UMBRAL)
auroc_val_O = auroc(PRED["val"], PRED["O"])
exactitud_val = accuracy_score(PRED["val"].etiqueta, PRED["val"].prediccion)


def ganancia_marginal(tabla, umbrales):
    d = tabla.set_index("umbral").loc[umbrales]
    delta_o = d.O_rechazado_pct.diff().iloc[1:]
    delta_v = d.rechazadas_pct.diff().iloc[1:]
    return pd.DataFrame({
        "tramo": [f"{a:.2f}→{b:.2f}" for a, b in zip(umbrales[:-1], umbrales[1:])],
        "hasta": umbrales[1:],
        "delta_O_pct": delta_o.to_numpy(),
        "delta_validas_pct": delta_v.to_numpy(),
        "ratio": (delta_o / delta_v).to_numpy(),
    })


def lectura_regla(gm):
    """Óptimo estricto (final del tramo de mayor ratio) y último umbral con ratio > 1 desde el inicio."""
    optimo = gm.loc[gm.ratio.idxmax(), "hasta"]
    positivos = gm.ratio.gt(1).cumprod().astype(bool)
    ultimo = gm.loc[positivos, "hasta"].max() if positivos.any() else np.nan
    return optimo, ultimo


UG = CFG_E1["umbrales_ganancia_marginal"]
gm_C1, gm_val = ganancia_marginal(curva_C1, UG), ganancia_marginal(curva_val, UG)
regla = {k: lectura_regla(g) for k, g in {"C1": gm_C1, "val": gm_val}.items()}

display(pd.concat({"C1 (test)": gm_C1.set_index("tramo").drop(columns="hasta"),
                   "val": gm_val.set_index("tramo").drop(columns="hasta")}, axis=1).round(2))

comparativa_085 = pd.DataFrame({
    "C1 (test)": [exactitud_C1, en_085_C1.cobertura, en_085_C1.exactitud_aceptadas,
                  en_085_C1.errores_capturados, en_085_C1.aciertos_perdidos, en_085_C1.O_rechazado_pct / 100,
                  auroc_C1_O, *regla["C1"]],
    "val": [exactitud_val, en_085_val.cobertura, en_085_val.exactitud_aceptadas,
            en_085_val.errores_capturados, en_085_val.aciertos_perdidos, en_085_val.O_rechazado_pct / 100,
            auroc_val_O, *regla["val"]],
}, index=["exactitud sin umbral", f"cobertura en {UMBRAL}", f"exactitud aceptadas en {UMBRAL}",
          f"errores capturados en {UMBRAL}", f"aciertos perdidos en {UMBRAL}", f"O rechazado en {UMBRAL}",
          "AUROC frente a O", "regla: óptimo estricto", "regla: último con ratio > 1"])
comparativa_085.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for tabla, punto, nombre, color in [(curva_C1, en_085_C1, "C1 (test)", AZUL), (curva_val, en_085_val, "val", AQUA)]:
    ax.plot(tabla.cobertura * 100, tabla.exactitud_aceptadas, color=color, label=nombre)
    ax.plot(punto.cobertura * 100, punto.exactitud_aceptadas, "o", ms=8, color=color, mec="white", mew=2)
ax.set(xlabel="Cobertura (% aceptado)", ylabel="Exactitud sobre las aceptadas", xlim=(0, 102))
ax.set_title(f"Riesgo-cobertura: C1 frente a val (punto = umbral {UMBRAL})", loc="left")
ax.legend(loc="lower left")
guardar_figura(fig, "riesgo_cobertura_C1_val.png")

## 8. E1d · C2 (corpus de desarrollo, fuera de EVA)

El clasificador nunca ha visto C2 y el umbral no se eligió con él: es la comprobación **limpia** del
umbral y del cambio de dominio (concurso EVA → fotos nuevas).

- **Sin A1** se puede medir ya: reparto de etiquetas, % enviado a `otro` (con IC de Wilson) y la
  distribución de la confianza en C2 frente a C1 y O.
- **Con A1 completa:** exactitud con 6 etiquetas (incluida `otro`), matriz 6×6, exactitud sobre las
  aceptadas, y recall de `otro` (de las que A1 marca `otro`, cuántas rechaza el sistema).

In [ ]:
c2 = PRED["C2"]
n_otro_C2 = int((c2.etiqueta_sistema == "otro").sum())
pct_otro_C2 = 100 * n_otro_C2 / len(c2)
ic_otro_C2 = comun.intervalo_wilson(n_otro_C2, len(c2))
print(f"C2 enviado a «otro»: {n_otro_C2}/{len(c2)} = {pct_otro_C2:.1f} % "
      f"(IC 95 % {100 * ic_otro_C2[0]:.1f}-{100 * ic_otro_C2[1]:.1f} %)")

conf_resumen = pd.DataFrame({k: PRED[k].confianza.describe(percentiles=[0.1, 0.25, 0.5, 0.75]) for k in ["C1", "O", "C2"]}).T
conf_resumen["% < umbral"] = [100 * (PRED[k].confianza < UMBRAL).mean() for k in conf_resumen.index]
display(conf_resumen.drop(columns="count").round(3))
display(pd.crosstab(c2.prediccion, c2.etiqueta_sistema, margins=True, margins_name="total"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.linspace(0.2, 1.0, 33)
for k, nombre, color in [("C1", "C1 (EVA test)", AZUL), ("O", "O (EVA other)", NARANJA), ("C2", "C2 (desarrollo)", AQUA)]:
    ax.hist(PRED[k].confianza, bins=bins, density=True, histtype="step", lw=2, color=color, label=nombre)
ax.axvline(UMBRAL, color=GRIS, lw=1, ls="--")
ax.text(UMBRAL, ax.get_ylim()[1] * 0.95, f" umbral {UMBRAL}", fontsize=8, color=TINTA2, va="top")
ax.set(xlabel="Confianza (probabilidad máxima)", ylabel="Densidad")
ax.set_title("Distribución de la confianza: EVA frente a C2", loc="left")
ax.legend(loc="upper left")
guardar_figura(fig, "confianza_C1_O_C2.png")

In [ ]:
E1d_A1 = {"estado": f"pendiente: A1 con {N_A1}/{len(C2)} imágenes anotadas"}
if A1_COMPLETA:
    d = c2.merge(A1[["imagen", "contexto"]], left_on="id", right_on="imagen")
    assert len(d) == len(c2)
    acepta = d.etiqueta_sistema != "otro"
    verdad_otro = d.contexto == "otro"
    etiquetas6 = comun.ETIQUETAS_A1
    cm_C2 = confusion_matrix(d.contexto, d.etiqueta_sistema, labels=etiquetas6)
    k6 = int((d.contexto == d.etiqueta_sistema).sum())
    k_acept = int((d.contexto == d.etiqueta_sistema)[acepta].sum())
    k5 = int((d.contexto == d.prediccion)[~verdad_otro].sum())
    E1d_A1 = {
        "estado": "calculado",
        "reparto_A1": d.contexto.value_counts().to_dict(),
        "exactitud_6_etiquetas": k6 / len(d),
        "ic_exactitud_6_etiquetas": comun.intervalo_wilson(k6, len(d)),
        "n_aceptadas": int(acepta.sum()),
        "exactitud_aceptadas": k_acept / max(int(acepta.sum()), 1),
        "ic_exactitud_aceptadas": comun.intervalo_wilson(k_acept, int(acepta.sum())),
        "exactitud_sin_umbral_5_clases": k5 / max(int((~verdad_otro).sum()), 1),
        "recall_otro": float((~acepta & verdad_otro).sum() / max(int(verdad_otro.sum()), 1)),
        "matriz_6x6": cm_C2,
        "etiquetas_matriz": etiquetas6,
    }
    dibujar_matriz(cm_C2, etiquetas6, etiquetas6,
                   f"C2 · con umbral {UMBRAL} · exactitud {E1d_A1['exactitud_6_etiquetas']:.3f}", "matriz_C2.png")
    display(d[d.contexto != d.etiqueta_sistema][["id", "contexto", "etiqueta_sistema", "prediccion", "confianza"]])
{k: v for k, v in E1d_A1.items() if k not in ("matriz_6x6",)}

## 9. E1e · Fidelidad del Nivel 1 (opcional, tras T100)

El orquestador es un LLM que **copia** la etiqueta del clasificador en `SalidaOrquestador`. Se comprueba
que `salida_orquestador.json:etiqueta_contexto` coincide con la etiqueta de `predict()` en cada ejecución
de la tanda T100 (`outputs/experimentos/T100/{rep}/{stem}/`).

In [ ]:
salidas_T100 = sorted((comun.ruta_config("tandas") / "T100").glob("*/imagen*/salida_orquestador.json"))
E1e = {"estado": "pendiente: no hay ejecuciones de la tanda T100"}
if salidas_T100:
    esperada = PRED["C2"].set_index("id").etiqueta_sistema
    filas = []
    for s in salidas_T100:
        copiada = json.loads(s.read_text(encoding="utf-8"))["etiqueta_contexto"]
        filas.append({"rep": s.parent.parent.name, "id": s.parent.name, "orquestador": copiada,
                      "predict": esperada.get(s.parent.name)})
    fidelidad = pd.DataFrame(filas)
    fidelidad["coincide"] = fidelidad.orquestador == fidelidad.predict
    E1e = {"estado": "calculado", "n": len(fidelidad), "coinciden": int(fidelidad.coincide.sum()),
           "discrepancias": fidelidad[~fidelidad.coincide].to_dict("records")}
    fidelidad.to_csv(RES / "fidelidad_orquestador_T100.csv", index=False, encoding="utf-8")
E1e

## 10. Grad-CAM del par más confundido

Par de C1 con más confusiones: **producto → arquitectura** (18 casos). Se elige de forma
**determinista**, sin mirar los mapas:

- el **acierto** de producto con mayor confianza;
- el **error** producto → arquitectura con mayor confianza. Es el caso más grave: el umbral no lo captura
  si supera 0,85.

El mapa se calcula para la clase predicha, con la misma capa que el sistema (`layer4[-1]`). Se superpone
sobre el recorte que ve la red (Resize 256 → CenterCrop 224).

Nota: `predict()` superpone el mapa sobre la imagen entera redimensionada a 224×224. En imágenes no
cuadradas, eso lo desalinea respecto al recorte.

In [ ]:
par = CFG_E1["gradcam_par"]
de_la_clase = c1[c1.etiqueta == par["verdadera"]]
errores_par = de_la_clase[de_la_clase.prediccion == par["confundida"]].sort_values("confianza", ascending=False)
aciertos_par = de_la_clase[de_la_clase.prediccion == par["verdadera"]].sort_values("confianza", ascending=False)
assert len(errores_par) == cm_C1[CLASES.index(par["verdadera"]), CLASES.index(par["confundida"])]
recorte = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])


def gradcam(ruta, clase):
    img = cc._abrir_para_clasificar(Path(ruta))
    x = clf.transform(img).unsqueeze(0).to(clf.device)
    mapa = clf.cam(input_tensor=x, targets=[ClassifierOutputTarget(CLASES.index(clase))])[0]
    base = np.asarray(recorte(img), dtype=np.float32) / 255.0
    return base, show_cam_on_image(base, mapa, use_rgb=True)


CASOS_CAM = {"acierto": aciertos_par.iloc[0], "error": errores_par.iloc[0]}
fig, axs = plt.subplots(2, 2, figsize=(7, 7.4))
for fila, (tipo, caso) in zip(axs, CASOS_CAM.items()):
    base, superpuesta = gradcam(EVA_IMG / f"{caso.id}.jpg", caso.prediccion)
    decision = "aceptada" if caso.confianza >= UMBRAL else "→ otro"
    fila[0].imshow(base)
    fila[0].set_title(f"{tipo} · EVA {caso.id}\nverdadera: {abreviar(caso.etiqueta)}", loc="left", fontsize=9)
    fila[1].imshow(superpuesta)
    fila[1].set_title(f"Grad-CAM de «{abreviar(caso.prediccion)}»\nconfianza {caso.confianza:.3f} ({decision})",
                      loc="left", fontsize=9)
    for ax in fila:
        ax.axis("off")
fig.tight_layout()
guardar_figura(fig, "gradcam_producto_arquitectura.png")
print(f"{len(errores_par)} errores del par; aceptados con confianza ≥ {UMBRAL}: {(errores_par.confianza >= UMBRAL).sum()}")

## 11. Criterio

El resultado se calcula (sí / no / pendiente); no se escribe a mano.

In [ ]:
def evaluar(valor, umbral, sentido, pendiente=None):
    if pendiente:
        return "pendiente", pendiente
    if umbral is None:
        return "pendiente", "criterio sin fijar en config_experimentos.yaml"
    ok = valor >= umbral if sentido == ">=" else valor <= umbral
    return ("sí" if ok else "no"), ""


f1_por_clase = {k: reporte_C1[k]["f1-score"] for k in CLASES}
f1_min = min(f1_por_clase.values())
filas = []
r, nota = evaluar(f1_min, CRIT["f1_min_por_clase_C1"], ">=")
filas.append(("F1 mínima por clase en C1", ">=", CRIT["f1_min_por_clase_C1"], f1_min, r,
              f"descriptivo (conocido de antemano); mínima: {min(f1_por_clase, key=f1_por_clase.get)}"))
r, nota = evaluar(E1d_A1.get("exactitud_aceptadas"), CRIT["exactitud_aceptadas_min_C2"], ">=",
                  None if A1_COMPLETA else E1d_A1["estado"])
filas.append(("Exactitud sobre aceptadas en C2", ">=", CRIT["exactitud_aceptadas_min_C2"],
              E1d_A1.get("exactitud_aceptadas"), r, nota))
r, nota = evaluar(pct_otro_C2, CRIT["pct_otro_max_C2"], "<=")
filas.append(("% de C2 enviado a «otro»", "<=", CRIT["pct_otro_max_C2"], pct_otro_C2, r, nota))
CRITERIOS = pd.DataFrame(filas, columns=["criterio", "sentido", "umbral", "valor", "resultado", "nota"])
CRITERIOS

## 12. Exportación

Todo va a `experimentos/resultados/E1/`:

- `predicciones_{C1,O,val,C2}.csv` (ya escritos en §4);
- `curva_C1_O.csv` y `curva_val_O.csv`;
- `metricas.json` (con la procedencia);
- las figuras PNG.

In [ ]:
curva_C1.to_csv(RES / "curva_C1_O.csv", index=False, encoding="utf-8")
curva_val.to_csv(RES / "curva_val_O.csv", index=False, encoding="utf-8")

metricas = {
    "experimento": "E1",
    "procedencia": PROC,
    "congelado": CONGELADO,
    "umbral_otro": UMBRAL,
    "replica_predict": {"n": len(chk), "dif_max_confianza": dif_conf, "tolerancia_tf32": TOL_TF32,
                        "cerca_del_umbral": {k: v.to_dict("records") for k, v in cerca_umbral.items()}},
    "E1a_C1": {
        "n": len(c1), "exactitud": exactitud_C1, "ic95_exactitud": ic_exactitud_C1,
        "f1_por_clase": f1_por_clase, "f1_macro": reporte_C1["macro avg"]["f1-score"],
        "precision_por_clase": {k: reporte_C1[k]["precision"] for k in CLASES},
        "recall_por_clase": {k: reporte_C1[k]["recall"] for k in CLASES},
        "matriz": cm_C1, "etiquetas_matriz": CLASES,
        "confusiones_principales": confusiones.to_dict("records"),
        "integridad_fase5": True,
    },
    "E1b_C1_O": {"auroc": auroc_C1_O, "en_umbral": en_085_C1.to_dict(), "n_errores_C1": n_err_C1},
    "E1c_val_O": {"exactitud_sin_umbral": exactitud_val, "auroc": auroc_val_O, "en_umbral": en_085_val.to_dict(),
                  "ganancia_marginal_C1": gm_C1.to_dict("records"), "ganancia_marginal_val": gm_val.to_dict("records"),
                  "regla_C1": dict(zip(["optimo_estricto", "ultimo_ratio_mayor_1"], regla["C1"])),
                  "regla_val": dict(zip(["optimo_estricto", "ultimo_ratio_mayor_1"], regla["val"]))},
    "E1d_C2": {"n": len(c2), "n_otro": n_otro_C2, "pct_otro": pct_otro_C2, "ic95_otro": ic_otro_C2,
               "reparto_etiqueta_sistema": c2.etiqueta_sistema.value_counts().to_dict(),
               "confianza_mediana": {k: float(PRED[k].confianza.median()) for k in ["C1", "O", "C2"]},
               "con_A1": E1d_A1},
    "E1e_orquestador": E1e,
    "gradcam": {t: {"id": int(c.id), "verdadera": c.etiqueta, "predicha": c.prediccion,
                    "confianza": float(c.confianza)} for t, c in CASOS_CAM.items()},
    "criterios": CRITERIOS.to_dict("records"),
}
comun.guardar_json(metricas, RES / "metricas.json")
sorted(p.name for p in RES.iterdir())

## 13. Limitaciones propias de E1 (para §8.6 y §9.1)

- **El umbral 0,85 se eligió mirando C1 y O.** La cobertura y la exactitud sobre las aceptadas en C1 son
  optimistas. E1c (val) y E1d (C2) son los controles.
- **O es *near-OOD*:** es el `other` de EVA (`sort = 6`), del mismo concurso y con la misma estética, y
  parte de sus imágenes sí tienen contenido de las 5 clases. Además, el `other` de EVA no es el `otro` del
  sistema (§5.1.2).
- **val no está del todo limpio:** con él se eligió el checkpoint (época), aunque no el umbral.
- **C2 lo etiqueta un único anotador** (A1), y es pequeño (n = 100): las proporciones llevan IC de Wilson.
- **El criterio F1 ≥ 0,75 es descriptivo:** la F1 mínima (0,7687, producto) se conocía antes de fijarlo.
- **Grad-CAM es ilustrativo:** dos casos elegidos con una regla fija. No demuestra en qué se fija la red en
  general.